In [ ]:
import os, re, glob
import pandas as pd
from datetime import datetime, timedelta, date
from zoneinfo import ZoneInfo
import numpy as np

NY = ZoneInfo("America/New_York")
UTC = ZoneInfo("UTC")

PARQUET_DIR = "parquet_daily"
GTFS_ROOT   = "gtfs/" 
OUT_DIR     = "clean_parquet"
TOLERANCE_MIN = 12

os.makedirs(OUT_DIR, exist_ok=True)
GTFS_CACHE = {}

def load_gtfs_tables(gtfs_dir: str):
    if gtfs_dir in GTFS_CACHE:
        return GTFS_CACHE[gtfs_dir]

    stop_times = pd.read_csv(os.path.join(gtfs_dir, "stop_times.txt"),
                             dtype={"trip_id":str,"stop_id":str,"arrival_time":str,"departure_time":str})
    trips = pd.read_csv(os.path.join(gtfs_dir, "trips.txt"),
                        dtype={"trip_id":str,"route_id":str,"service_id":str})
    calendar_df = pd.read_csv(os.path.join(gtfs_dir, "calendar.txt"),
                              dtype={"service_id":str, "start_date":str, "end_date":str})

    cal_dates_path = os.path.join(gtfs_dir, "calendar_dates.txt")
    calendar_dates_df = (pd.read_csv(cal_dates_path, dtype={"service_id":str, "date":str, "exception_type":str})
                         if os.path.exists(cal_dates_path) else None)

    stops = pd.read_csv(os.path.join(gtfs_dir, "stops.txt"), dtype=str)

    # Pre-join stop_times + trips once for faster lookup 
    st = stop_times.merge(trips[["trip_id","route_id","service_id"]], on="trip_id", how="inner")

    GTFS_CACHE[gtfs_dir] = {
        "st": st,
        "calendar_df": calendar_df,
        "calendar_dates_df": calendar_dates_df,
        "stops": stops
    }
    return GTFS_CACHE[gtfs_dir]


In [3]:
def date_from_part2_filename(path: str) -> date:
    # expects YYYY-MM-DD in filename like 2025-01-01_part2.parquet
    m = re.search(r"(\d{4}-\d{2}-\d{2})", os.path.basename(path))
    if not m:
        raise ValueError(f"Could not parse YYYY-MM-DD from: {path}")
    return datetime.strptime(m.group(1), "%Y-%m-%d").date()

def route_from_trip_uid(trip_uid: str) -> str:
    # 1735707600_7..S  -> "7"
    # 1735707600_GS.N01R -> "GS"
    try:
        after = str(trip_uid).split("_", 1)[1]
        return after.split(".", 1)[0]
    except Exception:
        return ""

def unix_to_ny(x):
    if pd.isna(x):
        return pd.NaT
    return datetime.fromtimestamp(int(x), tz=UTC).astimezone(NY)

def gtfs_time_to_dt(service_day: date, t: str):
    """Convert GTFS HH:MM:SS (can exceed 24:00:00) to NY datetime on service_day."""
    if pd.isna(t) or t == "":
        return None
    hh, mm, ss = t.split(":")
    h, m, s = int(hh), int(mm), int(ss)
    base = datetime(service_day.year, service_day.month, service_day.day, tzinfo=NY)
    return base + timedelta(hours=h, minutes=m, seconds=s)

def service_ids_for_date(calendar_df: pd.DataFrame, calendar_dates_df: pd.DataFrame | None, d: date) -> set[str]:
    ymd = int(d.strftime("%Y%m%d"))
    weekday_col = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"][d.weekday()]

    cal = calendar_df.copy()
    cal["start_date"] = cal["start_date"].astype(int)
    cal["end_date"] = cal["end_date"].astype(int)

    active = cal[
        (cal["start_date"] <= ymd) &
        (cal["end_date"] >= ymd) &
        (cal[weekday_col].astype(int) == 1)
    ]["service_id"]
    active_set = set(active.tolist())

    if calendar_dates_df is not None and len(calendar_dates_df) > 0:
        cd = calendar_dates_df.copy()
        cd["date"] = cd["date"].astype(int)
        todays = cd[cd["date"] == ymd][["service_id","exception_type"]]
        added = set(todays[todays["exception_type"].astype(int) == 1]["service_id"].tolist())
        removed = set(todays[todays["exception_type"].astype(int) == 2]["service_id"].tolist())
        active_set |= added
        active_set -= removed

    return active_set

In [4]:
def read_gtfs_range(gtfs_dir: str):
    cal = pd.read_csv(os.path.join(gtfs_dir, "calendar.txt"), dtype=str)
    start = int(cal["start_date"].min())
    end   = int(cal["end_date"].max())
    start_d = datetime.strptime(str(start), "%Y%m%d").date()
    end_d   = datetime.strptime(str(end), "%Y%m%d").date()
    return start_d, end_d

gtfs_index = []
for folder in sorted(os.listdir(GTFS_ROOT)):
    gtfs_dir = os.path.join(GTFS_ROOT, folder)
    if not os.path.isdir(gtfs_dir):
        continue
    if not os.path.exists(os.path.join(gtfs_dir, "calendar.txt")):
        continue
    s, e = read_gtfs_range(gtfs_dir)
    gtfs_index.append({"start": s, "end": e, "dir": gtfs_dir})

gtfs_index = pd.DataFrame(gtfs_index).sort_values(["start","end"]).reset_index(drop=True)
display(gtfs_index)

def pick_gtfs_for_day(d: date) -> str | None:
    # candidates that cover d
    cand = gtfs_index[(gtfs_index["start"] <= d) & (d <= gtfs_index["end"])].copy()
    if cand.empty:
        return None
    # overlap tie-break: latest start date wins (most recent schedule)
    best = cand.sort_values(["start","end"], ascending=[False, False]).iloc[0]
    return best["dir"]

,start,end,dir
0,2024-12-15,2025-01-17,gtfs/2024-12-12
1,2025-01-18,2025-05-18,gtfs/2025-1-16
2,2025-03-23,2025-05-18,gtfs/2025-03-24
3,2025-05-19,2025-06-07,gtfs/2025-05-15
4,2025-06-08,2025-11-01,gtfs/2025-06-05
5,2025-06-23,2025-11-01,gtfs/2025-06-21
6,2025-07-07,2025-11-01,gtfs/2025-07-04
7,2025-08-11,2025-11-01,gtfs/2025-08-08
8,2025-08-11,2025-11-01,gtfs/2025-08-11
9,2025-08-11,2025-11-01,gtfs/2025-08-14


In [5]:
KEYS = ["route_id","stop_id","event_kind","service_date"]

def groupwise_asof(left_df: pd.DataFrame, right_df: pd.DataFrame, tol: pd.Timedelta) -> pd.DataFrame:
    right_groups = {}
    for k, g in right_df.groupby(KEYS, sort=False):
        right_groups[k] = g.sort_values("scheduled_dt").reset_index(drop=True)

    out = []
    for k, g in left_df.groupby(KEYS, sort=False):
        g = g.sort_values("actual_dt").reset_index(drop=True)
        rg = right_groups.get(k)
        if rg is None or rg.empty:
            gg = g.copy()
            gg["scheduled_dt"] = pd.NaT
            gg["trip_id"] = pd.NA
            gg["stop_sequence"] = pd.NA
            out.append(gg)
            continue

        m = pd.merge_asof(
            g,
            rg,
            left_on="actual_dt",
            right_on="scheduled_dt",
            tolerance=tol,
            direction="nearest",
            allow_exact_matches=True,
            suffixes=("", "_sched"),
        )
        out.append(m)

    return pd.concat(out, ignore_index=True)

In [ ]:
def process_one_day(actual_parquet_path: str, gtfs: dict, tolerance_min: int = 12) -> pd.DataFrame:
    tol = pd.Timedelta(minutes=tolerance_min)

    actual = pd.read_parquet(actual_parquet_path)[["trip_uid","stop_id","arrival_time","departure_time"]].copy()
    # Vectorized route extraction (much faster than apply, equivalent to route_from_trip_uid)
    # Handles edge cases: splits on "_", takes [1], splits on ".", takes [0], fills "" if missing
    route_parts = actual["trip_uid"].astype(str).str.split("_", n=1, expand=True)
    actual["route_id"] = ""
    if len(route_parts.columns) > 1:
        actual["route_id"] = route_parts[1].str.split(".", n=1, expand=True)[0].fillna("")
    
    def safe_unix_to_dt(ser, tz):
        arr = pd.to_numeric(ser, errors="coerce")

        # kill infinities early
        arr = arr.replace([np.inf, -np.inf], np.nan)

        # pandas datetime64[ns] bounds in seconds/ms (UTC)
        max_s = pd.Timestamp.max.value / 1e9
        min_s = pd.Timestamp.min.value / 1e9
        max_ms = max_s * 1000.0
        min_ms = min_s * 1000.0

        absarr = arr.abs()


        ms_mask = absarr > 1e11  

        out = pd.Series(pd.NaT, index=arr.index, dtype="datetime64[ns, UTC]")

        #seconds
        s = arr.where(~ms_mask)
        s_ok = s.notna() & (s >= min_s) & (s <= max_s)

        with np.errstate(over="ignore", invalid="ignore"):
            out.loc[s_ok] = pd.to_datetime(s.loc[s_ok], unit="s", utc=True, errors="coerce")

        #ms
        ms = arr.where(ms_mask)
        ms_ok = ms.notna() & (ms >= min_ms) & (ms <= max_ms)

        with np.errstate(over="ignore", invalid="ignore"):
            out.loc[ms_ok] = pd.to_datetime(ms.loc[ms_ok], unit="ms", utc=True, errors="coerce")

        return out.dt.tz_convert(tz)
    actual["actual_arr_dt"] = safe_unix_to_dt(actual["arrival_time"], NY)
    actual["actual_dep_dt"] = safe_unix_to_dt(actual["departure_time"], NY)

    arr = actual[actual["actual_arr_dt"].notna()].copy()
    arr["event_kind"] = "arrival"
    arr["actual_dt"] = arr["actual_arr_dt"]

    dep = actual[actual["actual_dep_dt"].notna()].copy()
    dep["event_kind"] = "departure"
    dep["actual_dt"] = dep["actual_dep_dt"]

    actual_events = pd.concat([arr, dep], ignore_index=True)[["trip_uid","route_id","stop_id","event_kind","actual_dt"]]
    actual_events["service_date_0"] = actual_events["actual_dt"].dt.date
    actual_events["service_date_1"] = actual_events["service_date_0"].apply(lambda d: d - timedelta(days=1))

    dates_needed = set(actual_events["service_date_0"]) | set(actual_events["service_date_1"])

    st = gtfs["st"]
    calendar_df = gtfs["calendar_df"]
    calendar_dates_df = gtfs["calendar_dates_df"]
    stops = gtfs["stops"]

    sched_frames = []
    
    for d in sorted(dates_needed):
        active_sids = service_ids_for_date(calendar_df, calendar_dates_df, d)
        st_d = st[st["service_id"].isin(active_sids)].copy()
        if st_d.empty:
            continue

        #Vectorized GTFS time conversion (handles >24:00:00 correctly, matches original logic)
        base_dt = datetime(d.year, d.month, d.day, tzinfo=NY)
        
        #Parse HH:MM:SS - split and convert to timedelta (handles >24 hours, empty strings)
        def parse_gtfs_time_vectorized(time_str_series):
            #Filter out empty/NaN values first (matches original gtfs_time_to_dt logic)
            valid_mask = time_str_series.notna() & (time_str_series != "")
            result = pd.Series(index=time_str_series.index, dtype="timedelta64[ns]")
            
            if valid_mask.any():
                parts = time_str_series[valid_mask].str.split(":", expand=True).astype(int)
                #hours can be > 24 in GTFS - calculate total seconds
                total_seconds = parts[0] * 3600 + parts[1] * 60 + parts[2]
                result[valid_mask] = pd.to_timedelta(total_seconds, unit="s")
            
            return result
        
        arr_times = st_d["arrival_time"].copy()
        dep_times = st_d["departure_time"].copy()
        
        sarr = st_d.copy()
        sarr["event_kind"] = "arrival"
        arr_deltas = parse_gtfs_time_vectorized(arr_times)
        sarr["scheduled_dt"] = base_dt + arr_deltas
        sarr = sarr[sarr["scheduled_dt"].notna()]

        sdep = st_d.copy()
        sdep["event_kind"] = "departure"
        dep_deltas = parse_gtfs_time_vectorized(dep_times)
        sdep["scheduled_dt"] = base_dt + dep_deltas
        sdep = sdep[sdep["scheduled_dt"].notna()]

        sched = pd.concat([sarr, sdep], ignore_index=True)
        sched["service_date"] = d

        sched_frames.append(
            sched[["service_date","route_id","trip_id","stop_id","stop_sequence","event_kind","scheduled_dt"]]
        )

    if not sched_frames:
        return pd.DataFrame()

    schedule_events = pd.concat(sched_frames, ignore_index=True)
    schedule_events["service_date"] = pd.to_datetime(schedule_events["service_date"]).dt.date

    #Match using service_date_0 then fill with service_date_1 
    right = schedule_events.copy()
    right["scheduled_dt"] = pd.to_datetime(right["scheduled_dt"])

    left0 = actual_events.rename(columns={"service_date_0": "service_date"}).copy()
    left0["service_date"] = pd.to_datetime(left0["service_date"]).dt.date
    left0["actual_dt"] = pd.to_datetime(left0["actual_dt"])

    m0 = groupwise_asof(left0, right, tol)
    m0["matched_using"] = "service_date_0"

    need = m0["scheduled_dt"].isna()
    if need.any():
        left1 = actual_events.rename(columns={"service_date_1": "service_date"}).copy()
        left1["service_date"] = pd.to_datetime(left1["service_date"]).dt.date
        left1["actual_dt"] = pd.to_datetime(left1["actual_dt"])

        m1 = groupwise_asof(left1, right, tol)
        m1["matched_using"] = "service_date_1"

        keycols = ["trip_uid","route_id","stop_id","event_kind","actual_dt"]
        m0["k"] = m0[keycols].astype(str).agg("|".join, axis=1)
        m1["k"] = m1[keycols].astype(str).agg("|".join, axis=1)
        
        # Vectorized fill: merge m1 data where m0 has missing scheduled_dt
        m0_needs_fill = m0["scheduled_dt"].isna()
        if m0_needs_fill.any():
            # Create lookup from m1 (only rows with valid scheduled_dt)
            m1_valid = m1[m1["scheduled_dt"].notna()].set_index("k")
            
            # Get rows that need filling and can be filled
            fill_mask = m0_needs_fill & m0["k"].isin(m1_valid.index)
            if fill_mask.any():
                # Deduplicate m1_valid by "k" so loc returns 1 row per key
                # (merge_asof can produce duplicate keys; loc[duplicate_keys] returns too many rows)
                m1_valid_uniq = m1_valid[~m1_valid.index.duplicated(keep="first")]
                fill_keys = m0.loc[fill_mask, "k"]
                m1_fill = m1_valid_uniq.loc[fill_keys]
                
                # Update m0 with m1 values (vectorized)
                m0.loc[fill_mask, "scheduled_dt"] = m1_fill["scheduled_dt"].values
                m0.loc[fill_mask, "matched_using"] = m1_fill["matched_using"].values
                if "trip_id" in m0.columns:
                    m0.loc[fill_mask, "trip_id"] = m1_fill["trip_id"].values
                if "stop_sequence" in m0.columns:
                    m0.loc[fill_mask, "stop_sequence"] = m1_fill["stop_sequence"].values
        
        joined = m0.drop(columns=["k"])
    else:
        joined = m0

    # Calculate delay_seconds - ensure both columns are datetime64[ns] for proper subtraction
    joined["actual_dt"] = pd.to_datetime(joined["actual_dt"], errors="coerce")
    joined["scheduled_dt"] = pd.to_datetime(joined["scheduled_dt"], errors="coerce")
    # Subtract datetime columns - result is TimedeltaSeries, convert to seconds
    delay_td = joined["actual_dt"] - joined["scheduled_dt"]
    # Convert Timedelta to seconds (handles NaT as NaN)
    joined["delay_seconds"] = delay_td / pd.Timedelta(seconds=1)

    stops_small = stops[["stop_id","stop_name","parent_station","stop_lat","stop_lon"]].copy()
    joined = joined.merge(stops_small, on="stop_id", how="left")

    joined["direction"] = joined["stop_id"].str[-1].map({"N":"Uptown/North","S":"Downtown/South"})

    route_map = {"GS":"Grand Central Shuttle", "FS":"Franklin Av Shuttle", "H":"Rockaway Park Shuttle"}
    joined["route_name"] = joined["route_id"].replace(route_map)

    joined["service_date_out"] = pd.to_datetime(joined["scheduled_dt"]).dt.date
    joined["scheduled_time"] = pd.to_datetime(joined["scheduled_dt"]).dt.strftime("%H:%M:%S")
    joined["actual_time"] = pd.to_datetime(joined["actual_dt"]).dt.strftime("%H:%M:%S")
    joined["hour"] = pd.to_datetime(joined["scheduled_dt"]).dt.hour
    joined["day_of_week"] = pd.to_datetime(joined["scheduled_dt"]).dt.day_name()

    parent_lookup = stops[stops["location_type"].fillna("").astype(str).eq("1")][["stop_id","stop_name"]].copy()
    parent_lookup = parent_lookup.rename(columns={"stop_id":"parent_station","stop_name":"parent_stop_name"})
    joined = joined.merge(parent_lookup, on="parent_station", how="left")

    out = joined[[
        "service_date_out","route_name","direction","stop_id","stop_name","parent_stop_name",
        "stop_sequence","event_kind","scheduled_dt","actual_dt","delay_seconds",
        "scheduled_time","actual_time","hour","day_of_week","trip_uid"
    ]].copy()

    return out.rename(columns={"service_date_out":"service_date"})


In [ ]:
stop_level_files = sorted(glob.glob(os.path.join(PARQUET_DIR, "*_part2.parquet")))
print("Stop-level files found:", len(stop_level_files))

skipped_no_gtfs = []
written = 0

for idx, p in enumerate(stop_level_files, 1):
    d = date_from_part2_filename(p)
    print(f"[{idx}/{len(stop_level_files)}] Processing {d}...", end=" ", flush=True)
    
    gtfs_dir = pick_gtfs_for_day(d)

    if gtfs_dir is None:
        skipped_no_gtfs.append((d, os.path.basename(p)))
        print("SKIP (no GTFS)")
        continue

    out_path = os.path.join(OUT_DIR, f"{d}.parquet")
    if os.path.exists(out_path):
        print("SKIP (already exists)")
        continue

    print("loading GTFS...", end=" ", flush=True)
    gtfs = load_gtfs_tables(gtfs_dir)
    print("processing...", end=" ", flush=True)

    day_df = process_one_day(p, gtfs, tolerance_min=TOLERANCE_MIN)

    if day_df.empty:
        print(f"WARN (empty output)")
        continue

    print(f"saving...", end=" ", flush=True)
    day_df.to_parquet(out_path, index=False)
    written += 1
    print(f"OK ({len(day_df):,} rows)")

print(f"\nDone! Written: {written}")
if skipped_no_gtfs:
    print("Days with no GTFS coverage:")
    print(skipped_no_gtfs[:10])


In [ ]:
import os, glob
import pandas as pd
from collections import defaultdict

metric_col = "delay_seconds"

KEY_COLS = [
    "route_name",
    "direction",
    "stop_id",
    "stop_name",
    "stop_sequence",
    "hour"
]

clean_files = sorted(glob.glob(os.path.join(OUT_DIR, "*.parquet")))

if not clean_files:
    raise RuntimeError("No cleaned parquet files found in OUT_DIR.")

NEW_OUT_DIR = os.path.join(OUT_DIR, "month_weekday_hour")
os.makedirs(NEW_OUT_DIR, exist_ok=True)

stats = defaultdict(lambda: defaultdict(lambda: [0.0, 0]))

print("Processing daily files...")
for idx, f in enumerate(clean_files, 1):
    if idx % 50 == 0:
        print(f"{idx}/{len(clean_files)}")

    df = pd.read_parquet(f)

    df["service_date"] = pd.to_datetime(df["service_date"])
    df["month_period"] = df["service_date"].dt.to_period("M").astype(str)
    df["day_of_week"] = df["service_date"].dt.day_name()

    # arrivals only
    df = df[df["event_kind"] == "arrival"]

    # valid delays
    df = df[df[metric_col].notna()]
    if df.empty:
        continue

    for (mp, dow, *key_vals), g in df.groupby(
        ["month_period", "day_of_week"] + KEY_COLS, observed=True
    ):
        key = tuple(key_vals)
        bucket = stats[(mp, dow)][key]
        bucket[0] += g[metric_col].sum()
        bucket[1] += len(g)

print("Writing datasets to new folder...")

for (mp, dow), key_map in stats.items():
    rows = []
    for key, (total, count) in key_map.items():
        row = {"month_period": mp, "day_of_week": dow}
        row.update({col: val for col, val in zip(KEY_COLS, key)})
        row["avg_delay_seconds"] = total / count
        row["count"] = count
        rows.append(row)

    out_df = pd.DataFrame(rows)

    out_df = out_df[out_df["count"] >= 4]

    out_path = os.path.join(NEW_OUT_DIR, f"{mp}_{dow}.parquet")
    out_df.to_parquet(out_path, index=False)

print("Saved to:", NEW_OUT_DIR)


Processing daily files...
50/365
100/365
150/365
200/365
250/365
300/365
350/365
Writing datasets to new folder...
✅ Done.
Saved to: clean_parquet/month_weekday_hour
